In [ ]:
import numpy as np            # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绘图库
import cv2 as cv              # OpenCV计算机视觉库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图
        plt.imshow(img, cmap='gray')
    else:  # 彩色图，BGR转RGB
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.show()

## 1. 卷积

In [ ]:
img = np.ones((5,5))    # 创建5x5全1矩阵，模拟简单图像
kernel = np.ones((3,3)) # 创建3x3全1卷积核

print(img)
print(kernel)

In [ ]:
# filter2D：对图像进行二维卷积操作
# 参数：输入图像, 输出图像深度(-1表示与输入相同), 卷积核
# 3x3全1核相当于累加邻域3x3=9个像素值，每个输出都是9
img2 = cv.filter2D(img, -1, kernel)
print(img2)

## 2. 均值滤波

In [ ]:
img = cv.imread('pic/rose_spnoise_200x200.jpg')  # 读取含椒盐噪声的玫瑰图像
show(img)

In [ ]:
# 均值滤波：卷积核为全1/(3*3)=1/9，对邻域像素取平均
K = np.ones((3,3)) / 9  # 3x3均值核，每个权重为1/9

# filter2D手动实现均值滤波：用自定义核进行二维卷积
img1 = cv.filter2D(img, -1, K)
show(np.hstack([img, img1]))  # 水平拼接原图与滤波结果进行对比

In [ ]:
# cv.blur：OpenCV内置均值滤波函数，效果等同于filter2D + 均值核
# 参数：输入图像, 卷积核大小(width, height)
img2 = cv.blur(img, (3,3))
show(np.hstack([img, img2]))

In [ ]:
# cv.boxFilter：盒子滤波，normalize=True时等同于均值滤波
# 参数：输入图像, 输出深度(-1同源), 核大小, normalize是否归一化
img3 = cv.boxFilter(img, -1, (3,3))
show(np.hstack([img, img3]))

## 3. 中值模糊

In [ ]:
# cv.medianBlur：中值滤波，取邻域内所有像素值的中位数
# 对椒盐噪声效果极佳，因为椒盐噪声是极端值，中值运算可有效剔除
# 参数：输入图像, 核大小（必须为奇数）
img4 = cv.medianBlur(img, 3)
show(np.hstack([img, img4]))

## 4. 高斯滤波

In [ ]:
img = cv.imread('pic/rose_spnoise_200x200.jpg')  # 重新读取含噪图像
show(img)

In [ ]:
sigma = 0.8  # 高斯函数的标准差，控制模糊程度

# cv.GaussianBlur：高斯滤波，权重服从二维高斯分布（中心权重最大）
# 参数：输入图像, 核大小(5,5), sigmaX=X方向标准差
# sigmaX越小，核权重越集中于中心，模糊效果越弱
img2 = cv.GaussianBlur(img, (5,5), sigmaX=sigma)
show(img2)

## 5. 双边滤波

In [ ]:
img = cv.imread('pic/beer.jpg', 0)  # 以灰度模式读取啤酒图像（0=IMREAD_GRAYSCALE）
show(img)

In [ ]:
# cv.bilateralFilter：双边滤波，同时考虑空间距离和像素值差异
# 在平滑区域降噪的同时，能保留边缘细节（边缘两侧像素值差异大，权重小，不被平滑）
# 参数：输入图像, 滤波直径(-1自动计算), sigmaColor(颜色空间sigma), sigmaSpace(坐标空间sigma)
img2 = cv.bilateralFilter(img, -1, sigmaColor=50, sigmaSpace=3)
show(img2)

## 6. 双边滤波实现

$$
c(\xi - x) = e^{-0.5(\frac{\lVert \xi-x \rVert}{\sigma_d})^2}
$$

In [ ]:
# 空间权重函数：基于像素间欧氏距离的高斯衰减
# 公式：c(xi - x) = exp(-0.5 * (||xi - x|| / sigma_d)^2)
# 距离中心越远，权重越小（空间距离衰减）
def get_C(sigmad, n):
    C = np.zeros((n,n))
    
    x = np.array([n//2, n//2])  # 中心点坐标
    for i in range(n):
        for j in range(n):
            ksi = np.array([i, j])
            # norm计算欧式距离，离中心越远指数衰减越大
            C[i,j] = np.exp(-0.5 * (np.linalg.norm(ksi - x) / sigmad)**2)
            
    C /= C.sum()  # 归一化，使所有权重之和为1
    return C

In [ ]:
# 用sigma_d=3, 核大小11x11生成空间权重矩阵，可视化高斯衰减形状
C = get_C(3, 11)
show(C)

$$
s(f(\xi)- f(x)) = e^{-0.5(\frac{\lVert f(\xi)-f(x) \rVert}{\sigma_r})^2}
$$

In [ ]:
# 像素值权重函数（循环版本）：基于邻域与中心像素值差异的高斯衰减
# 公式：s(f(xi)- f(x)) = exp(-0.5 * (|f(xi)-f(x)| / sigma_r)^2)
# 像素值越接近中心像素，权重越大（保边原理的关键）
def get_S(f, sigmar, n):
    S = np.zeros((n,n))
    
    f = np.float64(f)  # 转为float64防止uint8溢出
    for i in range(n):
        for j in range(n):
            # 像素值差越大，指数衰减越大，权重越小
            S[i,j] = np.exp(-0.5 * ((f[i,j] - f[n//2, n//2]) / sigmar)**2)
            
    S /= S.sum()  # 归一化
    return S

In [ ]:
# 像素值权重函数（向量化版本）：与get_S功能相同，但用NumPy广播代替循环，效率更高
def get_S2(f, sigmar, n):
    f = np.float64(f)  # 转为float64防止计算溢出
    
    # 广播：f - f[n//2,n//2] 计算所有像素与中心的差值，一次性计算全部权重
    S = np.exp(-0.5 * ((f - f[n//2, n//2]) / sigmar)**2)
            
    S /= S.sum()  # 归一化
    return S

In [ ]:
# 用空间权重矩阵C作为输入，sigma_r=10计算像素值权重
# C中心值最大、边缘最小，经过S变换后形状类似（因为输入值差异小）
S1 = get_S(C, 10, 11)
show(S1)

In [ ]:
# 向量化版本验证：结果应与get_S完全一致
S2 = get_S2(C, 10, 11)
show(S2)

In [ ]:
img = cv.imread('pic/beer.jpg', 0)  # 灰度模式读取啤酒图（手动实现双边滤波的输入）
show(img)

In [ ]:
sigmar = 50   # 像素值空间的sigma，控制保边强度（越大越接近均值滤波）
sigmad = 3    # 坐标空间的sigma，控制空间衰减速度
n = 11        # 滤波核大小（11x11邻域）

h, w = img.shape    # 获取图像高宽
img2 = np.zeros_like(img)  # 创建与原图同尺寸的空输出图像

In [ ]:
%%time
# 手动实现双边滤波（逐像素循环，速度较慢，用于理解原理）

C = get_C(sigmad, n)  # 空间权重矩阵（固定，只计算一次）

# 遍历每个像素（跳过边缘n/2像素，避免越界）
for i in range(h-n):
    for j in range(w-n):
        f = img[i:i+n, j:j+n]                          # 提取当前像素的11x11邻域
        S = get_S(f, sigmar, n)                         # 计算该邻域的像素值权重
        K = C * S                                       # 最终权重 = 空间权重 * 像素值权重（逐元素相乘）
        K /= K.sum()                                    # 归一化
        img2[i,j] = (f * K).sum()                       # 加权求和得到输出像素值

show(img2)

cv2.bilateralFilter(src,d,sigmaColor,sigmaSpace,borderType)
- sigmaColor：表示在滤波处理时选取的颜色差值范围，该值决定了周围哪些像素点能够参与到滤波中来。与当前像素点的像素值差值小于sigmaColor的像素点，能够参与到当前的滤波中。该值越大，就说明周围有越多的像素点可以参与到运算中。该值为0时，滤波失去意义；该值为255时，指定直径内的所有点都能够参与运算。
- sigmaSpace：表示坐标空间中的sigma值。它的值越大，说明有越多的点能够参与到滤波计算中来。当时，无论sigmaSpace的值如何，d都指定邻域大小；否则，d与sigmaSpace的值成比例。


In [ ]:
%%time
# cv.bilateralFilter：OpenCV内置双边滤波（C++加速，速度远快于手动实现）
# d=11为滤波直径，sigmaColor=50对应手写的sigmar，sigmaSpace=3对应sigmad
img3 = cv.bilateralFilter(img, 11, sigmaColor=50, sigmaSpace=3)
show(img3)

In [ ]:
# 将原图、手动实现结果、OpenCV结果水平拼接并保存为uint8格式
# astype(np.uint8)：clip到0~255并转为8位无符号整数，imwrite要求此格式
cv.imwrite('test/bilateral.jpg', np.hstack([img, img2, img3]).astype(np.uint8))